# Delta-V budget — how it works

This is the explanation and reference for the `quicksat` delta-V budget. It covers what the manoeuvre file holds, and which closed forms turn each row into a delta-V. It also explains how its single margin differs from the mass budget's two layers, and where the tool stops.

It is not a tutorial, on purpose. In [Diátaxis](https://diataxis.fr/) terms, `sample/` holds the tutorials and how-to guides, and `docs/` holds the explanation and the reference. It still runs, because an explanation that cannot run soon stops matching the code it describes. It runs against `docs/sizing/data/`, its own copy of the sample satellite's input files. The copy is on purpose. The figures quoted below are true only of the data that produced them. If this notebook read `sample/sizing/data/`, a change to the tutorial's inputs would change the figures here without raising an error.

One premise holds throughout: **Mission Analysis does the orbital mechanics.** This module holds their numbers, and turns each into a delta-V with a short closed form. Anything that needs real analysis comes as a `given` manoeuvre, with the answer already worked out.

In [1]:
import os
import tempfile
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import Q_, R_EARTH, u
from quicksat.delta_v.budget import (
    DeltaVBudget,
    deorbit_deltav,
    hohmann_deltav,
    plane_change_deltav,
)
from quicksat.utils.mission import Mission

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

DATA = Path("docs") / "sizing" / "data"
mission = Mission.from_yaml_file(DATA / "mission.yaml")
budget = DeltaVBudget.from_csv(
    DATA / "manoeuvres.csv", DATA / "delta_v_config.yaml", DATA / "mission.yaml"
)

## The data model

This budget is a table, like the mass budget and unlike the data budget. There is a list of manoeuvres, and the budget is their sum. So the input is a CSV, and `phase` and `manoeuvre_type` are two ordinary columns to group over.

What differs from the mass budget is how the margin is applied. There is **one margin on the total**, not a contingency per item plus a system margin. The reason is where the uncertainty lies. A Hohmann transfer's delta-V comes from a closed form, and is known to more figures than anyone needs. *How many collision avoidance manoeuvres a year* is an estimate that could be wrong by a factor of two. A contingency on each manoeuvre would add margin to the precise part, and none to the uncertain part.

In [2]:
budget.manoeuvre_table

,manoeuvre_id,manoeuvre_name,phase,manoeuvre_type,value,count,recurring,loss_factor,comments
0,injection_correction,Launcher dispersion correction,Commissioning,altitude_change,12.0 km,1.000,False,1.000,Semi-major axis dispersion at separation
1,inclination_trim,Injection inclination trim,Commissioning,inclination_change,0.05 deg,1.000,False,1.000,Plane error at separation
2,phasing,Orbit phasing to the reference slot,Commissioning,given,8.0 m / s,1.000,False,1.000,From Mission Analysis; needs a real phasing an...
3,drag_makeup,Drag make-up,Operations,altitude_change,1.2 km,1.000,True,1.000,Per year at 500 km solar mean
4,collision_avoidance,Collision avoidance,Operations,collision_avoidance,200.0 m,4.000,True,1.000,Per year; one-way hop as the drag make-up abso...
5,inclination_maint,Inclination maintenance,Operations,inclination_change,0.012 deg,1.000,True,1.000,"Per year, holds the sun-synchronous plane"
6,deorbit,End-of-life deorbit,Disposal,deorbit,250.0 km,1.000,False,1.000,Single burn to a 500 x 250 km disposal orbit; ...


## The manoeuvre file

| column | meaning |
|---|---|
| `manoeuvre_id` | short identifier, no whitespace (`collision_avoidance`) |
| `manoeuvre_name` | full name, free text |
| `phase` | mission phase — the reporting axis (`Commissioning`, `Operations`, `Disposal`) |
| `manoeuvre_type` | which closed form applies |
| `value` | the input, with its unit; what it *means* depends on the type |
| `count` | how many times |
| `recurring` | when set, `count` is per year and is multiplied by the mission duration |
| `loss_factor` | optional, default 1.0 — finite-burn and gravity losses, as a multiplier |
| `comments` | free text |

### One value column, five meanings

A manoeuvre's input is a length for an altitude change, and an angle for a plane change. It is a velocity for a delta-V that Mission Analysis computed elsewhere. Rather than four mostly empty columns, there is one `value` column. The type decides which dimension it must have:

| type | `value` is | delta-V |
|---|---|---|
| `altitude_change` | altitude delta | Hohmann between the two circular altitudes |
| `collision_avoidance` | altitude offset | the same Hohmann, doubled when the config says the hop returns |
| `inclination_change` | angle | `2·v·sin(Δi/2)` |
| `deorbit` | target perigee altitude | one impulse: `v·(1 − √(2r_p/(r+r_p)))` |
| `given` | the delta-V itself | taken as-is |

This check is the main reason for merging four columns into one. It is a model validator rather than a field validator, because the required dimension depends on another field in the same row:

In [3]:
HEADER = "manoeuvre_id,manoeuvre_name,phase,manoeuvre_type,value,count,recurring,comments"


def load_row(label, row):
    """Build a one-row budget from a CSV line, and report what the loader makes of it."""
    path = Path(tempfile.mkdtemp()) / "manoeuvres.csv"
    path.write_text(f"{HEADER}\n{row}\n")
    try:
        DeltaVBudget.from_csv(path, DATA / "delta_v_config.yaml", DATA / "mission.yaml")
        print(f"{label:34s} accepted")
    except ValueError as exc:
        reason = str(exc).split("Value error, ")[-1].split(" [type=")[0]
        print(f"{label:34s} rejected - {reason}")


load_row("a plain altitude change", "x,X,Ops,altitude_change,1 km,1,false,")
load_row("velocity where a length belongs", "x,X,Ops,altitude_change,8 m/s,1,false,")
load_row("length where a velocity belongs", "x,X,Ops,given,200 m,1,false,")
load_row("length where an angle belongs", "x,X,Ops,inclination_change,5 km,1,false,")
load_row("an angle, correctly", "x,X,Ops,inclination_change,0.05 deg,1,false,")
load_row("a whitespace id", "a b,X,Ops,given,8 m/s,1,false,")
load_row("a negative count", "x,X,Ops,given,8 m/s,-1,false,")

a plain altitude change            accepted
velocity where a length belongs    rejected - A 'altitude_change' manoeuvre needs a length in its value column, got '8.0 m / s'
length where a velocity belongs    rejected - A 'given' manoeuvre needs a velocity in its value column, got '200.0 m'
length where an angle belongs      rejected - A 'inclination_change' manoeuvre needs an angle in its value column, got '5.0 km'
an angle, correctly                accepted
a whitespace id                    rejected - /tmp/tmp732w4csn/manoeuvres.csv, row 2:
1 validation error for Manoeuvre
manoeuvre_id
  String should match pattern '^\S+$'
a negative count                   rejected - /tmp/tmpcxov1gei/manoeuvres.csv, row 2:
1 validation error for Manoeuvre
count
  Input should be greater than or equal to 0


## The closed forms

There are three, and all assume circular orbits and impulsive burns.

**Hohmann transfer** between two circular radii. It takes two impulses, one to leave and one to circularise:

$$\Delta v = v_1\left|\sqrt{\tfrac{2r_2}{r_1+r_2}} - 1\right| + v_2\left|1 - \sqrt{\tfrac{2r_1}{r_1+r_2}}\right|$$

**Plane change** at circular velocity. This is why it needs so much delta-V: $\Delta v = 2v\sin(\Delta i / 2)$.

**Deorbit**, a single impulse that lowers the perigee. The new perigee is either at a disposal altitude, where drag brings the satellite down, or at the surface, for a direct re-entry: $\Delta v = v\left(1 - \sqrt{\tfrac{2r_p}{r+r_p}}\right)$.

In [4]:
print(f"circular velocity at {mission.altitude:.0f}:  {mission.velocity:.4f}")
print()
for delta in ("200 m", "1 km", "12 km"):
    hop = hohmann_deltav(mission.radius, mission.radius + Q_(delta))
    print(f"  raise by {delta:6s}  {hop:8.4f}")

disposal = deorbit_deltav(mission.radius, R_EARTH + 250 * u.km)
reentry = deorbit_deltav(mission.radius, R_EARTH)
print(f"\ndeorbit to a 250 km perigee  {disposal:.2f}")
print(f"deorbit to the surface       {reentry:.2f}")
print("  the same single impulse; the target perigee is the biggest lever there is")

# a plane change is priced at orbital velocity, so a fraction of a degree is dear
trim = plane_change_deltav(mission.velocity, 0.05 * u.deg)
print(f"\n0.05 deg of plane change {trim:.3f}")
print(f"  which is what raising the orbit by 12 km costs "
      f"({hohmann_deltav(mission.radius, mission.radius + 12 * u.km):.3f})")

circular velocity at 500 km:  7.6126 km / s

  raise by 200 m     0.1107 m / s
  raise by 1 km      0.5533 m / s
  raise by 12 km     6.6321 m / s

deorbit to a 250 km perigee  70.78 m / s
deorbit to the surface       144.95 m / s
  the same single impulse; the target perigee is the biggest lever there is

0.05 deg of plane change 6.643 m / s
  which is what raising the orbit by 12 km costs (6.632 m / s)


### The closed forms are impulsive, and `loss_factor` corrects them

Every formula above assumes that the burn happens instantly, at one point. A real burn takes time. During it, the thrust does not point where an impulsive manoeuvre would need it, and the vehicle keeps falling. So the delta-V needed is larger than the closed form says.

This module cannot work out how much larger. It depends on the thrust level, on the burn arc, and most of all on how the manoeuvre is split. A plane change or a deorbit is usually flown as several burns at successive apses, rather than as one long burn. Each burn's arc sets its loss. That is an operational choice, so it comes as an input.

`loss_factor` multiplies a row's delta-V. 1.0 is the impulsive ideal and the default. So a manoeuvre file without the column gives exactly the same budget as before. 1.03 asks for 3% more. The factor cannot be below 1.0, because a burn cannot do better than the impulsive ideal. So 0.97 must be a typo. It raises an error on load, rather than reducing the budget without a warning.

**It is not the margin, and neither replaces the other.** `loss_factor` is a systematic correction to a manoeuvre whose size is known. The margin on the total covers the manoeuvres whose *counts* are estimates. Using one to cover doubt about the other gives a number that is hard to defend later.

### Collision avoidance has no physics of its own

A collision avoidance manoeuvre raises the orbit by a few hundred metres. The spacecraft waits for the conjunction to pass, and then comes back down. That is two Hohmann transfers. So the type reuses the altitude change, and multiplies it by two.

The config's `return_burn` turns the doubling off. This is a real operational case, not a modelling shortcut. The satellite may already be due for a raise to make up for drag. The avoidance manoeuvre then serves as that raise, and there is nothing to undo.

In [5]:
one_way = hohmann_deltav(mission.radius, mission.radius + 200 * u.m)
frame = budget.resolve().set_index("manoeuvre_id")

print(f"one-way 200 m hop        {one_way:.4f}")
charged = frame.loc["collision_avoidance", "deltav_each"]
print(f"the budget's CAM line    {charged:.4f} m/s")
print(
    f"  return_burn is {budget.config.collision_avoidance.return_burn}, "
    f"so it is charged {charged / one_way.value:.0f}x the one-way hop"
)

one-way 200 m hop        0.1107 m / s
the budget's CAM line    0.1107 m/s
  return_burn is False, so it is charged 1x the one-way hop


## Counts, and the mission duration

`count` is how many times a manoeuvre happens. When `recurring` is set, the count is a **rate**: that many per year. The mission duration turns it into a total. Without `recurring`, the count is the total for the whole mission.

This is why the duration is a separate number, rather than being built into the counts. Extending a 7 year mission to 10 years is then one edit, not a change to every row.

It is in `mission.yaml` rather than in this budget's config, because other budgets use the design life too. The data budget can report totals over it, and the power and radiator work will need it. `mission_ref.ipynb` gives the rule that decides which file a fact belongs in.

In [6]:
resolved = budget.resolve()
resolved[["manoeuvre_id", "count", "recurring", "occurrences", "deltav_each", "deltav_total"]]

,manoeuvre_id,count,recurring,occurrences,deltav_each,deltav_total
0,injection_correction,1.000,False,1.000,6.632,6.632
1,inclination_trim,1.000,False,1.000,6.643,6.643
2,phasing,1.000,False,1.000,8.000,8.000
3,drag_makeup,1.000,True,7.000,0.664,4.648
4,collision_avoidance,4.000,True,28.000,0.111,3.099
5,inclination_maint,1.000,True,7.000,1.594,11.161
6,deorbit,1.000,False,1.000,70.784,70.784


## The margin

The margin is one allowance, applied once, to the total. It is applied last. So both grouped views include it in proportion, and still add up to the same number.

In [7]:
for margin in (False, True):
    total = budget.total_deltav(margin)
    phases = budget.by_phase(margin)["deltav"].sum()
    types = budget.by_type(margin)["deltav"].sum()
    label = f"margin={margin}"
    print(f"{label:14s} total {total:8.2f}   by phase {phases:8.2f}   by type {types:8.2f}")

print(f"\nthe margin is {budget.config.margin:g}% of the budget, or "
      f"{(budget.total_deltav() - budget.total_deltav(False)):.2f}")

margin=False   total   110.97 m / s   by phase   110.97   by type   110.97
margin=True    total   116.52 m / s   by phase   116.52   by type   116.52

the margin is 5% of the budget, or 5.55 m / s


## The rocket equation, and the sizing loop

In the end, a delta-V budget gives a propellant mass. Given the dry mass and the exhaust velocity $v_e = I_{sp}g_0$:

$$m_{prop} = m_{dry}\left(e^{\Delta v / v_e} - 1\right)$$

The argument is the **dry** mass, which is the mass after the burn. So the propellant is what has to be added to it. Passing the wet mass instead would answer a different question, and give a smaller number.

A `MassBudget` can be attached when the delta-V budget is built. `propellant_mass()` then takes the dry mass from it. This is the in-orbit mass with the tanks empty, which is the mass after the last burn. A mass passed as an argument overrides it. Passing a mass is the only way to use the method when nothing is attached.

The attachment is optional, and goes one way only. `quicksat.delta_v` never imports `quicksat.mass` at runtime, because the reference is a `TYPE_CHECKING` annotation. So the delta-V budget still loads, resolves and reports with no spacecraft attached. Only this one method needs one. Nothing is written back in either direction. The equipment CSV stays the only record of the propellant actually loaded, and a person compares the two.

In [8]:
from quicksat.mass.budget import MassBudget

mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "mass_budget_config.yaml")
dry = mass_data.in_orbit_mass(propellant=0)

# the same manoeuvres, with the spacecraft attached
flown = DeltaVBudget.from_csv(
    DATA / "manoeuvres.csv",
    DATA / "delta_v_config.yaml",
    DATA / "mission.yaml",
    mass_budget=mass_data,
)

print(f"dry mass             {dry:.2f}   (taken from the attached budget)")
print(f"delta-V              {flown.total_deltav():.2f}")
print(f"propellant required  {flown.propellant_mass():.2f}")
print(f"propellant loaded    {mass_data.propellant_mass():.2f}")
print()
print("the two disagree, which is the loop doing its job: the equipment list was")
print("written before the delta-V budget existed, and has not been revised to match")
print()

# an explicit mass overrides the attachment, which is how a what-if is asked
# without touching the equipment file -- and the only way in with nothing attached
print("what a heavier spacecraft would need, without editing anything:")
for extra in (0, 50, 100):
    what_if = dry + extra * u.kg
    print(f"  dry {what_if:7.2f}  ->  {flown.propellant_mass(what_if):6.2f}")

dry mass             448.62 kg   (taken from the attached budget)
delta-V              116.52 m / s
propellant required  24.89 kg
propellant loaded    22.00 kg

the two disagree, which is the loop doing its job: the equipment list was
written before the delta-V budget existed, and has not been revised to match

what a heavier spacecraft would need, without editing anything:
  dry  448.62 kg  ->   24.89 kg
  dry  498.62 kg  ->   27.67 kg
  dry  548.62 kg  ->   30.44 kg


### One burn or a hundred, the propellant is the same

Summing the delta-V column and applying the rocket equation once to the total looks like a shortcut. It is not. It is exact, and it stays exact at any propellant fraction.

Each burn multiplies the mass by its own factor, $m_{before} = m_{after}\,e^{\Delta v_i / v_e}$. So a sequence of burns multiplies those factors together, and $e^{a}e^{b} = e^{a+b}$. The product reduces to a single exponential of the *sum* of the delta-Vs. Nothing in that step assumes that the burns are small, or that the propellant is a small fraction of the dry mass.

This needs saying, because intuition suggests otherwise. A burn late in the mission pushes less mass than the same burn at the start. So it should need less propellant, and it does. But the earlier burns had to carry that late burn's propellant. The saving is exactly the extra propellant that the earlier burns used to carry it, so the two cancel. The equation works backwards from the dry mass, so the cancellation happens automatically. The order of the burns does not matter either, because multiplication is commutative.

So the budget can treat seven manoeuvres spread over seven years as one number. A drag make-up flown once a year for seven years needs exactly the propellant of the same total flown as a single burn.

In [9]:
import numpy as np

# astropy.constants creates its constants at import, so pyright cannot see them
from astropy.constants import g0  # pyright: ignore[reportAttributeAccessIssue]

exhaust_velocity = budget.config.propulsion.isp * g0
margin_factor = 1 + budget.config.margin / 100


def staged_propellant(deltavs):
    """Propellant when each delta-V is flown as its own burn, worked backwards from dry."""
    mass = dry
    for deltav in deltavs:
        mass = mass * np.exp((deltav * u.m / u.s) / exhaust_velocity)
    return mass - dry


per_row = (budget.resolve()["deltav_total"] * margin_factor).to_numpy()
total = budget.total_deltav().value

print(f"all seven manoeuvres at once  {budget.propellant_mass(dry):.6f}")
print(f"one rocket equation per row   {staged_propellant(per_row):.6f}")
print(f"the same rows, reversed       {staged_propellant(per_row[::-1]):.6f}")
print(f"1000 equal instalments        {staged_propellant(np.full(1000, total / 1000)):.6f}")

all seven manoeuvres at once  24.894198 kg
one rocket equation per row   24.894198 kg
the same rows, reversed       24.894198 kg
1000 equal instalments        24.894198 kg


The identity also holds as the propellant grows. The sample satellite's propellant is 5% of its dry mass, where a difference would be easy to miss. The identity is just as exact at fifteen times the dry mass.

In [10]:
print(f"{'total ΔV':>12}  {'one burn':>13}  {'100 burns':>13}  {'propellant/dry':>14}")
for total_dv in (100, 500, 1500, 3000, 6000):
    one = dry * (np.exp((total_dv * u.m / u.s) / exhaust_velocity) - 1)
    many = staged_propellant(np.full(100, total_dv / 100))
    fraction = (one / dry).to_value(u.dimensionless_unscaled)
    print(f"{total_dv:>9} m/s  {one:>10,.4f}  {many:>10,.4f}  {fraction:>13.0%}")

    total ΔV       one burn      100 burns  propellant/dry
      100 m/s     21.2833 kg     21.2833 kg             5%
      500 m/s    117.0042 kg    117.0042 kg            26%
     1500 m/s    450.5187 kg    450.5187 kg           100%
     3000 m/s  1,353.4627 kg  1,353.4627 kg           302%
     6000 m/s  6,790.2502 kg  6,790.2502 kg          1514%


What would break the identity is not the number of burns. It is a change in the exhaust velocity, or in the dry mass:

- **A second propulsion system.** Two exhaust velocities do not reduce to one exponential. What matters then is how the delta-V *divides between the systems*. Moving part of the budget onto a thruster with a higher `isp` changes the propellant a lot. The order still does not matter. quicksat models a single `isp`, so this case is out of scope by design, not by oversight.
- **Hardware jettisoned between burns.** With staging, the dry mass is not one number. The chain has to be worked out one stage at a time, each with its own final mass.

Finite-burn losses need a word here. They are the one case where splitting a manoeuvre into several burns *does* change something. At first sight, this seems to contradict everything above.

It does not, because the two statements are about different quantities. The rocket equation does not depend on how a **given** delta-V is divided. This section has just shown that, exactly and at any mass fraction. But the delta-V passed to it is not fixed. As the `loss_factor` section says, a manoeuvre flown as several short arcs loses less to finite-burn effects than one long burn. So the split changes the loss factor. The loss factor changes the delta-V, and the delta-V changes the propellant.

So the two roles stay separate. How a manoeuvre is flown belongs in `loss_factor`, where it is visible and open to review. It never belongs in the mass arithmetic, which cannot see it and does not need to.

## The document view

`tabulated_deltav()` lays out the budget as a document. The manoeuvres are grouped into mission phases, each with a subtotal. Then come the total, the margin, and the total with margin. The phases keep the order of the file rather than being sorted. In a manoeuvre file, that order is usually chronological, so the table follows the mission.

It follows the same Styler conventions as the other two reports. Empty cells are blank rather than NaN, and summaries are bold. `row_type` is in `.data` but hidden in the display, and `comments` is hidden unless asked for. `loss_factor=True` works the same way. The column is always in `.data`, and shown only when asked for. In a budget of impulsive burns, it is a column of ones. Turn it on whenever a manoeuvre has a correction. A delta-V budget that includes finite-burn losses without showing them is hard to defend in a review.

In [11]:
budget.tabulated_deltav(loss_factor=True)

Item,Name,Type,Input,Loss factor,ΔV each [m/s],Times,ΔV total [m/s]
injection_correction,Launcher dispersion correction,altitude_change,12 km,1,6.632,1.0,6.63
inclination_trim,Injection inclination trim,inclination_change,0.05 deg,1,6.643,1.0,6.64
phasing,Orbit phasing to the reference slot,given,8 m / s,1,8.000,1.0,8.00
,Commissioning subtotal,,,,,,21.28
drag_makeup,Drag make-up,altitude_change,1.2 km,1,0.664,7.0,4.65
collision_avoidance,Collision avoidance,collision_avoidance,200 m,1,0.111,28.0,3.10
inclination_maint,Inclination maintenance,inclination_change,0.012 deg,1,1.594,7.0,11.16
,Operations subtotal,,,,,,18.91
deorbit,End-of-life deorbit,deorbit,250 km,1,70.784,1.0,70.78
,Disposal subtotal,,,,,,70.78


In [12]:
data = budget.tabulated_deltav().data
print(data["row_type"].value_counts().to_dict())

summary = data[data["row_type"].isin(["subtotal", "margin", "total"])]
print()
print(summary[["name", "deltav"]].to_string(index=False))  # pyright: ignore[reportAttributeAccessIssue]

{'manoeuvre': 7, 'phase_subtotal': 3, 'subtotal': 1, 'margin': 1, 'total': 1}

                name  deltav
Total, before margin 110.967
         Margin (5%)   5.548
            Total ΔV 116.515


## Limitations

What the delta-V budget does not do, on purpose:

- **Impulsive burns, corrected by hand.** The closed forms assume instant impulses. `loss_factor` is the only way that finite-burn and gravity losses enter, and you supply it rather than the tool deriving it. There is no thrust-to-weight check. So nothing warns you when a manoeuvre would need an implausible number of burns. This model does not cover a low-thrust electric transfer at all. That needs a different formulation, not a correction factor.
- **Circular orbits, two-impulse transfers.** There is no eccentricity anywhere. So there are no bi-elliptic transfers, no combined manoeuvres, and no plane change at apogee, where it would need less delta-V.
- **No perturbations.** The drag make-up figure is an input, not something computed from an atmosphere model and a ballistic coefficient. So is the inclination maintenance.
- **One propulsion system, one Isp.** There is no split between cold gas and monopropellant, no blowdown curve, and no thruster cant or cosine losses.
- **The rocket equation ignores staging and residuals.** There is no unusable propellant, no pressurant, and no margin for a failed burn that has to be repeated.
- **No epoch, no sequencing.** Manoeuvres have counts, not dates. Nothing models a conjunction rate that rises with the debris environment, or a deorbit that needs less delta-V because drag has already lowered the orbit.
- **The margin is a flat percentage.** It is not a statistical combination. It is applied to the total rather than to the uncertain parts. This is conservative, on purpose.